In [18]:
import pandas as pd

train_path = "/content/drive/MyDrive/Pose86K-CSLR-Isharah/annotations_v2/SI/train.txt"
dev_path   = "/content/drive/MyDrive/Pose86K-CSLR-Isharah/annotations_v2/SI/dev.txt"

df_train = pd.read_csv(train_path, sep="|")
df_dev   = pd.read_csv(dev_path, sep="|")

print(df_train.head())
print(df_dev.head())
print("Train samples:", len(df_train))
print("Dev samples:", len(df_dev))
import pickle

pkl_path   = "/content/drive/MyDrive/Pose86K-CSLR-Isharah/data/pose_data_isharah1000_hands_lips_body_May12.pkl"

with open(pkl_path, "rb") as f:
    pose_dict = pickle.load(f)

print("Total sequences in PKL:", len(pose_dict))
sample_key = next(iter(pose_dict.keys()))
print("Example sample shape:", pose_dict[sample_key]["keypoints"].shape)
# Collect all gloss tokens
all_gloss_tokens = []

for g in df_train["gloss"]:
    all_gloss_tokens.extend(g.split())

for g in df_dev["gloss"]:
    all_gloss_tokens.extend(g.split())

vocab = ["<BLANK>"] + sorted(set(all_gloss_tokens))

g2i = {g: i for i, g in enumerate(vocab)}
i2g = {i: g for g, i in g2i.items()}

print("Vocab size:", len(vocab))
print("Example tokens:", vocab[:20])
def encode_gloss_seq(sentence):
    return [g2i[token] for token in sentence.split()]

df_train["encoded"] = df_train["gloss"].apply(encode_gloss_seq)
df_dev["encoded"]   = df_dev["gloss"].apply(encode_gloss_seq)

df_train[["id","encoded"]].head()
import numpy as np

def load_pose(sample_id):
    if sample_id not in pose_dict:
        return None
    return pose_dict[sample_id]["keypoints"]   # (T, 86, 2)
X_train = []
Y_train = []
missing_train = []

for idx, row in df_train.iterrows():
    sample_id = row["id"]
    pose = load_pose(sample_id)

    if pose is None:
        missing_train.append(sample_id)
        continue

    X_train.append(pose)
    Y_train.append(row["encoded"])

print("Loaded train:", len(X_train))
print("Missing train:", missing_train[:10])
X_dev = []
Y_dev = []
missing_dev = []

for idx, row in df_dev.iterrows():
    sample_id = row["id"]
    pose = load_pose(sample_id)

    if pose is None:
        missing_dev.append(sample_id)
        continue

    X_dev.append(pose)
    Y_dev.append(row["encoded"])

print("Loaded dev:", len(X_dev))
print("Missing dev:", missing_dev[:10])
print("Final X_train:", len(X_train))
print("Final X_dev:", len(X_dev))
print("Final Y_train:", len(Y_train))
print("Final Y_dev:", len(Y_dev))
print("Vocabulary size:", len(vocab))


        id                 gloss                   text
0  00_0001               سوال هو                  من هو
1  00_0002     هو معلم لغه اشاره      هو مدرس لغه اشاره
2  00_0003    استفهام هو معلم هو            هل انت مدرس
3  00_0004  هو معلم لا انا مدرسه  انت لست مدرس انا طالب
4  00_0005               هو سوال                  هو من
        id                 gloss
0  02_0001               سوال هو
1  02_0002     هو معلم لغه اشاره
2  02_0003    استفهام هو معلم هو
3  02_0004  هو معلم لا انا مدرسه
4  02_0005               هو سوال
Train samples: 10000
Dev samples: 949


/tmp/ipython-input-718820652.py:18: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  pose_dict = pickle.load(f)


Total sequences in PKL: 10450
Example sample shape: (55, 86, 2)
Vocab size: 684
Example tokens: ['<BLANK>', 'ا', 'اب', 'ابتسامه', 'ابن', 'ابها', 'ابيض', 'اتصال', 'اتصالات', 'اتقان', 'اثنان', 'اثنان_ثلاثون', 'اثنين', 'اجازه', 'اجتماع', 'احد_عشر', 'احرام', 'احمر', 'اخ', 'اخبار']
Loaded train: 9500
Missing train: ['00_0451', '00_0452', '00_0453', '00_0454', '00_0455', '00_0456', '00_0457', '00_0458', '00_0459', '00_0460']
Loaded dev: 949
Missing dev: []
Final X_train: 9500
Final X_dev: 949
Final Y_train: 9500
Final Y_dev: 949
Vocabulary size: 684


In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")


Using device: cuda


In [29]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

class PoseDataset(Dataset):
    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        pose = self.X[idx]                    # (T, 86, 2)
        pose = pose.reshape(pose.shape[0], -1)  # → (T, 172)
        pose = torch.tensor(pose, dtype=torch.float32)

        label = torch.tensor(self.Y[idx], dtype=torch.long)
        return pose, label


def collate_fn(batch):
    poses, labels = zip(*batch)

    lengths = [p.shape[0] for p in poses]
    max_len = max(lengths)

    padded_poses = []
    for p in poses:
        T, D = p.shape
        if T < max_len:
            pad = torch.zeros(max_len - T, D)
            p = torch.cat([p, pad], dim=0)
        padded_poses.append(p)

    return torch.stack(padded_poses), list(labels), torch.tensor(lengths, dtype=torch.long)


In [30]:
train_dataset = PoseDataset(X_train, Y_train)
dev_dataset   = PoseDataset(X_dev, Y_dev)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=collate_fn
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=collate_fn
)

print("Train batches:", len(train_loader))
print("Dev batches:", len(dev_loader))


Train batches: 297
Dev batches: 949


In [31]:
import torch.nn.utils.rnn as rnn_utils

class BiGRU_CSLR(nn.Module):
    def __init__(self, input_dim=172, hidden_dim=512, num_layers=3, num_classes=684):
        super().__init__()

        self.gru = nn.GRU(
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            dropout=0.1,
            batch_first=True
        )

        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x, lengths):
        # PACK the padded sequence
        packed = rnn_utils.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )

        packed_out, _ = self.gru(packed)

        # UNPACK
        out, _ = rnn_utils.pad_packed_sequence(packed_out, batch_first=True)

        # Classification
        return self.classifier(out)


In [32]:
num_classes = len(vocab)

model_bigru = BiGRU_CSLR(
    input_dim=172,
    hidden_dim=512,
    num_layers=3,
    num_classes=num_classes
).to(device)

try:
    model_bigru = torch.compile(model_bigru)
    print("Model compiled!")
except:
    print("torch.compile not supported, continuing without it.")

criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model_bigru.parameters(), lr=2e-3, fused=True)


Model compiled!


In [33]:
def decode_argmax_ctc(logits):
    pred = logits.argmax(-1).cpu().tolist()
    decoded = []
    prev = -1
    for p in pred:
        if p != 0 and p != prev:
            decoded.append(p)
        prev = p
    return decoded

def edit_distance(a, b):
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        dp[i][0] = i
    for j in range(m+1):
        dp[0][j] = j

    for i in range(1,n+1):
        for j in range(1,m+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )
    return dp[-1][-1]

def evaluate_bigru(model, loader, device, i2g):
    model.eval()
    total_dist = 0
    total_words = 0
    example = None

    with torch.no_grad():
        for poses, labels, lengths in loader:
            poses = poses.to(device)
            lengths = lengths.to(device)

            logits = model(poses, lengths)[0]  # (T, C)

            pred_ids = decode_argmax_ctc(logits)
            pred_tokens = [i2g[i] for i in pred_ids]

            true_ids = labels[0].tolist()
            true_tokens = [i2g[i] for i in true_ids]

            total_dist += edit_distance(pred_tokens, true_tokens)
            total_words += len(true_tokens)

            if example is None:
                example = (pred_tokens, true_tokens)

    wer = total_dist / total_words
    return wer, example


In [34]:
from torch.amp import GradScaler, autocast

def train_bigru_epoch(model, loader, optimizer, criterion, device):
    model.train()
    scaler = GradScaler("cuda")
    total_loss = 0

    for poses, labels, lengths in loader:
        poses = poses.to(device)
        lengths = lengths.to(device)

        flat_targets = torch.cat(labels).to(device)
        target_lengths = torch.tensor(
            [len(l) for l in labels], dtype=torch.long, device=device
        )

        optimizer.zero_grad(set_to_none=True)

        with autocast("cuda"):
            logits = model(poses, lengths)
            log_probs = F.log_softmax(logits, dim=-1).permute(1,0,2)

            loss = criterion(log_probs, flat_targets, lengths, target_lengths)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


In [35]:
EPOCHS = 100

for epoch in range(1, EPOCHS+1):
    train_loss = train_bigru_epoch(model_bigru, train_loader, optimizer, criterion, device)
    wer, example = evaluate_bigru(model_bigru, dev_loader, device, i2g)

    print(f"Epoch {epoch:02d}: Train Loss={train_loss:.4f} | Dev WER={wer:.4f}")

    if example is not None and epoch % 5 == 0:
        pred, truth = example
        print("Sample GT glosses:   ", truth)
        print("Sample Pred glosses: ", pred)
        print("-" * 60)


Epoch 01: Train Loss=12.2482 | Dev WER=1.0000
Epoch 02: Train Loss=5.7955 | Dev WER=0.9194
Epoch 03: Train Loss=5.7208 | Dev WER=0.9194
Epoch 04: Train Loss=5.6597 | Dev WER=0.9016
Epoch 05: Train Loss=5.6284 | Dev WER=0.9106
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'سوال']
------------------------------------------------------------
Epoch 06: Train Loss=5.6195 | Dev WER=0.9561
Epoch 07: Train Loss=5.6015 | Dev WER=0.8996
Epoch 08: Train Loss=5.5915 | Dev WER=0.8996
Epoch 09: Train Loss=5.5630 | Dev WER=0.9708
Epoch 10: Train Loss=5.5541 | Dev WER=0.8996
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'سوال']
------------------------------------------------------------
Epoch 11: Train Loss=5.5404 | Dev WER=0.8996
Epoch 12: Train Loss=5.5346 | Dev WER=0.8964
Epoch 13: Train Loss=5.5287 | Dev WER=0.8946
Epoch 14: Train Loss=5.5167 | Dev WER=0.9708
Epoch 15: Train Loss=5.5096 | Dev WER=0.8996
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glo

KeyboardInterrupt: 

**Model description**

BiGRU Baseline Model

Input: Pose keypoints sequence (T, 86, 2) flattened to (T, 172)

Preprocessing: Frame padding/cropping + normalization

Encoder: 3-layer Bidirectional GRU (hidden size = 512)

Output: Linear layer to vocabulary logits to CTC Loss

Decoder: Greedy CTC decoding (collapse repeats + remove blanks)

Metric: Word Error Rate (WER)

**Training Output**



```
Epoch 01: Train Loss=12.2482 | Dev WER=1.0000
Epoch 02: Train Loss=5.7955 | Dev WER=0.9194
Epoch 03: Train Loss=5.7208 | Dev WER=0.9194
Epoch 04: Train Loss=5.6597 | Dev WER=0.9016
Epoch 05: Train Loss=5.6284 | Dev WER=0.9106
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'سوال']

Epoch 06: Train Loss=5.6195 | Dev WER=0.9561
Epoch 07: Train Loss=5.6015 | Dev WER=0.8996
Epoch 08: Train Loss=5.5915 | Dev WER=0.8996
Epoch 09: Train Loss=5.5630 | Dev WER=0.9708
Epoch 10: Train Loss=5.5541 | Dev WER=0.8996
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'سوال']

Epoch 11: Train Loss=5.5404 | Dev WER=0.8996
Epoch 12: Train Loss=5.5346 | Dev WER=0.8964
Epoch 13: Train Loss=5.5287 | Dev WER=0.8946
Epoch 14: Train Loss=5.5167 | Dev WER=0.9708
Epoch 15: Train Loss=5.5096 | Dev WER=0.8996
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'سوال']

Epoch 16: Train Loss=5.5148 | Dev WER=0.8977
Epoch 17: Train Loss=5.5153 | Dev WER=0.9260
Epoch 18: Train Loss=5.5170 | Dev WER=0.9559
Epoch 19: Train Loss=5.5054 | Dev WER=0.8996
Epoch 20: Train Loss=5.5012 | Dev WER=0.8961
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'هو']

Epoch 21: Train Loss=5.5020 | Dev WER=0.8996
Epoch 22: Train Loss=5.4992 | Dev WER=0.8996
Epoch 23: Train Loss=5.5062 | Dev WER=0.9073
Epoch 24: Train Loss=5.5201 | Dev WER=0.8946
Epoch 25: Train Loss=5.5066 | Dev WER=0.8959
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['انا', 'مع_السلامه']

Epoch 26: Train Loss=5.5065 | Dev WER=0.8992
Epoch 27: Train Loss=5.5039 | Dev WER=0.9001
Epoch 28: Train Loss=5.5125 | Dev WER=0.9003
Epoch 29: Train Loss=5.5121 | Dev WER=0.9001
Epoch 30: Train Loss=5.5114 | Dev WER=0.9014
Sample GT glosses:    ['سوال', 'هو']
Sample Pred glosses:  ['مع_السلامه']

**Test Results**

Final Dev WER = 0.9014

**Error analysis**

1- Severe Underfitting and Lack of Learning

Unlike the transformer model, which reduced its training loss aggressively, the BiGRU loss stagnates around 5.50 from epoch 5 onward. This suggests:

The model is not extracting meaningful temporal features from the raw 172-dimensional frame vectors.

The learning rate or architecture may be insufficient to capture patterns in the dataset.

The GRU struggles with very long unnormalized pose sequences (T often > 150-250 frames).

The dev WER staying flat around 0.90 indicates near-complete underfitting.

<br>
2- Dominant Error Type: Substitution Errors

Unlike the Transformer model (where insertions were dominant), the BiGRU primarily makes substitution errors:

It rarely predicts more than 1-2 glosses.

It usually outputs a single incorrect gloss (e.g., 'مع_السلامه').

It often replaces one or both ground-truth glosses with unrelated but common words.

This indicates that the BiGRU:

Does not learn temporal alignment across the sequence.

Collapses long sequences into a single global representation.

Behaves similarly to a bag of frames classifier instead of a sequence recognizer.

<br>
3- GRU Saturation on Long Sequences

Many Isharah sequences have 100-300 frames.

A 3-layer BiGRU processing these long sequences without normalization:

Experiences gradient decay across time.

Learns only early-frame or high-variance patterns.

Fails to preserve long-range dependencies needed for full-sentence recognition.

This explains why predictions often degenerate to a single gloss.